In [8]:
import os
import gc
import sys
import itertools
import torch
from tqdm.auto import tqdm
import pypdfium2 as pdfium
from colpali_engine.models import ColQwen2_5, ColQwen2Processor
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding

In [9]:
# 1. Cấu hình hệ thống & Chống phân mảnh CUDA
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

PDF_PATH = "trr2.pdf"
MODEL_ID = "vidore/colqwen2.5-v0.2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DPI = 150
DOC_BATCH_SIZE = 1
COLLECTION_NAME = "toan_roi_rac_hybrid"

forwards = [0.25, 0.30, 0.35, 0.40, 0.45, 0.48]
backwards = [0.05, 0.10, 0.15, 0.20, 0.25]

In [10]:




# 2. Tập dữ liệu Ground Truth 116 câu Multi-Page Ground Truth
test_set = [
    # CHƯƠNG 1
    {"query": "Định nghĩa đơn đồ thị vô hướng và ví dụ mạng máy tính loại 1", "target_pages": [7]},
    {"query": "Định nghĩa đa đồ thị vô hướng, cạnh bội và giả đồ thị chứa khuyên", "target_pages": [8]},
    {"query": "Bảng 1 phân biệt các loại đồ thị: vô hướng, có hướng, cạnh bội, khuyên", "target_pages": [9, 10]},
    {"query": "Định lý 1 về tổng bậc của các đỉnh bằng hai lần số cạnh và hệ quả số đỉnh bậc lẻ", "target_pages": [10, 11]},
    {"query": "Khái niệm đường đi đơn, chu trình đơn và định nghĩa đồ thị vô hướng liên thông", "target_pages": [11]},
    {"query": "Định nghĩa đỉnh trụ, cạnh cầu và ví dụ xác định cạnh cầu trên đồ thị vô hướng", "target_pages": [12]},
    {"query": "Bán bậc vào deg-(v) và bán bậc ra deg+(v) của đỉnh trên đồ thị có hướng", "target_pages": [13]},
    {"query": "Khái niệm đồ thị có hướng liên thông mạnh và liên thông yếu", "target_pages": [13, 14]},
    {"query": "Các dạng đồ thị đặc biệt: đồ thị đầy đủ Kn, đồ thị vòng Cn, đồ thị bánh xe Wn", "target_pages": [15]},

    # CHƯƠNG 2
    {"query": "Biểu diễn đồ thị vô hướng bằng ma trận kề và tính chất tổng hàng bằng bậc của đỉnh", "target_pages": [17, 18]},
    {"query": "Tính chất ma trận kề của đồ thị có hướng và ý nghĩa của lũy thừa ma trận A^p", "target_pages": [18, 19]},
    {"query": "Ma trận trọng số của đồ thị có hướng và ưu điểm của ma trận kề", "target_pages": [19]},
    {"query": "Nhược điểm của ma trận kề và quy ước khuôn dạng lưu trữ ma trận kề trong file dothi.in", "target_pages": [20]},
    {"query": "Biểu diễn đồ thị có hướng bằng danh sách cạnh và tính chất bậc ra bậc vào", "target_pages": [21, 22]},
    {"query": "Biểu diễn đồ thị trọng số bằng danh sách cạnh và ưu nhược điểm của danh sách cạnh", "target_pages": [22, 23]},
    {"query": "Cấu trúc dữ liệu struct Edge trong C biểu diễn danh sách cạnh bằng mảng", "target_pages": [23, 24]},
    {"query": "Biểu diễn đồ thị bằng danh sách kề Ke(u) và danh sách liên kết List(u)", "target_pages": [24, 25]},
    {"query": "Biểu diễn danh sách kề dựa vào mảng phân đoạn và mảng lưu vị trí VT[]", "target_pages": [25]},
    {"query": "Sơ đồ biểu diễn danh sách kề bằng danh sách liên kết và khuôn dạng file lưu trữ", "target_pages": [25, 26]},

    # CHƯƠNG 3
    {"query": "Tư tưởng cơ bản và thuật toán đệ qui DFS(u) duyệt theo chiều sâu", "target_pages": [31]},
    {"query": "Thuật toán DFS(u) khử đệ qui sử dụng ngăn xếp stack và độ phức tạp tính toán", "target_pages": [32]},
    {"query": "Bảng kiểm nghiệm các bước duyệt thuật toán DFS(1) trên đồ thị vô hướng 13 đỉnh", "target_pages": [33]},
    {"query": "Bảng 3.1 kiểm nghiệm trạng thái ngăn xếp stack và tập đỉnh duyệt DFS(1) từ ma trận kề", "target_pages": [34]},
    {"query": "Cài đặt thuật toán DFS_Dequi và DFS_Stack trong ngôn ngữ C", "target_pages": [35, 36]},
    {"query": "Biểu diễn thuật toán tìm kiếm theo chiều rộng BFS(u) sử dụng hàng đợi Queue", "target_pages": [37]},
    {"query": "Bảng 3.2 kiểm nghiệm trạng thái hàng đợi Queue và tập đỉnh được duyệt BFS(1)", "target_pages": [38, 39]},
    {"query": "Chương trình C cài đặt thuật toán BFS sử dụng mảng queue", "target_pages": [39, 40]},
    {"query": "Thuật toán Duyet-TPLT xác định các thành phần liên thông của đồ thị", "target_pages": [41, 42, 43]},
    {"query": "Thuật toán tìm đường đi giữa các đỉnh trên đồ thị sử dụng mảng truoc[]", "target_pages": [44]},
    {"query": "Mã giả thuật toán DFS và BFS tìm đường đi từ s đến t", "target_pages": [45]},
    {"query": "Thủ tục Ghi-Nhan-Duong-Di(s, t) và Bảng 3.3 kiểm nghiệm đường đi DFS", "target_pages": [45, 46]},
    {"query": "Thuật toán Strong-Connective kiểm tra tính liên thông mạnh của đồ thị có hướng", "target_pages": [49, 50]},
    {"query": "Bảng 3.5 kiểm nghiệm thuật toán kiểm tra tính liên thông mạnh trên từng đỉnh", "target_pages": [49, 50]},
    {"query": "Thuật toán Duyet-Tru xác định các đỉnh trụ của đồ thị vô hướng", "target_pages": [53]},
    {"query": "Bảng 3.6 kiểm nghiệm duyệt các đỉnh trụ của đồ thị qua phép duyệt DFS", "target_pages": [53, 54]},
    {"query": "Thuật toán Duyet-Cau liệt kê tất cả các cạnh cầu bằng cách tạm loại bỏ cạnh", "target_pages": [56, 57]},
    {"query": "Bảng 3.7 kiểm nghiệm duyệt các cạnh cầu và kết luận cạnh (3,5), (9,10) là cầu", "target_pages": [57, 58]},
    {"query": "Sơ đồ duyệt các thành phần liên thông mạnh và bài toán định chiều đồ thị vô hướng", "target_pages": [61, 62]},

    # CHƯƠNG 4
    {"query": "Định nghĩa chu trình Euler, đường đi Euler, đồ thị Euler và đồ thị nửa Euler", "target_pages": [68]},
    {"query": "Thuật toán Euler-Cycle(u) tìm chu trình Euler bằng ngăn xếp stack và mảng CE", "target_pages": [70]},
    {"query": "Bảng 4.1 kiểm nghiệm chi tiết các bước tìm chu trình Euler bắt đầu tại đỉnh 1", "target_pages": [71]},
    {"query": "Cài đặt chương trình C kiểm tra đồ thị Euler và hàm Euler-Cycle", "target_pages": [72]},
    {"query": "Định lý 2 và Định lý 3 về điều kiện để đồ thị vô hướng và có hướng là nửa Euler", "target_pages": [73]},
    {"query": "Thuật toán Euler-Path(u) tìm đường đi Euler xuất phát từ đỉnh bậc lẻ", "target_pages": [75]},
    {"query": "Bảng 4.2 kiểm nghiệm thuật toán tìm đường đi Euler trên đồ thị có hướng liên thông yếu", "target_pages": [75, 76]},
    {"query": "Định nghĩa đường đi Hamilton, chu trình Hamilton và thuật toán đệ quy quay lui Hamilton(k)", "target_pages": [78, 79]},
    {"query": "Hình 4.7 cây tìm kiếm chu trình Hamilton và code C cài đặt tìm chu trình Hamilton", "target_pages": [80, 81]},

    # CHƯƠNG 5
    {"query": "Định nghĩa cây, rừng và định lý về các khẳng định tương đương của một cây n đỉnh", "target_pages": [87, 88]},
    {"query": "Mô tả thuật toán Tree-DFS(u) xây dựng cây khung của đồ thị dựa vào tìm kiếm chiều sâu", "target_pages": [88, 89]},
    {"query": "Bảng 5.1 kiểm nghiệm quá trình kết nạp cạnh vào cây khung theo Tree-Graph-DFS", "target_pages": [89, 90]},
    {"query": "Thuật toán Tree-BFS(u) xây dựng cây khung của đồ thị dựa vào tìm kiếm chiều rộng", "target_pages": [91, 92, 93]},
    {"query": "Bảng 5.2 kiểm nghiệm các bước kết nạp cạnh vào cây khung theo thuật toán Tree-BFS", "target_pages": [92, 93]},
    {"query": "Phát biểu bài toán cây khung có độ dài nhỏ nhất và mô hình bài toán nối mạng máy tính", "target_pages": [95]},
    {"query": "Mô tả thuật toán Kruskal tìm cây khung nhỏ nhất và bước sắp xếp cạnh tăng dần", "target_pages": [96]},
    {"query": "Bảng kiểm nghiệm các bước lặp kết nạp cạnh của thuật toán Kruskal", "target_pages": [97, 98]},
    {"query": "Thuật toán Prim tìm cây bao trùm nhỏ nhất theo nguyên lý người láng giềng gần nhất", "target_pages": [100, 101]},
    {"query": "Bảng kiểm nghiệm các bước chọn cạnh nối giữa tập V và tập VH theo thuật toán Prim", "target_pages": [101, 102]},

    # CHƯƠNG 6
    {"query": "Phát biểu tổng quát bài toán tìm đường đi ngắn nhất từ đỉnh nguồn s đến đỉnh đích t", "target_pages": [107]},
    {"query": "Mô tả thuật toán Dijkstra tìm đường đi ngắn nhất từ đỉnh s với trọng số không âm", "target_pages": [107, 108]},
    {"query": "Bảng 6.1 kiểm nghiệm chi tiết các bước gán nhãn của thuật toán Dijkstra tại đỉnh 1", "target_pages": [109, 110]},
    {"query": "Mã nguồn C cài đặt thuật toán Dijkstra tìm đường đi ngắn nhất", "target_pages": [110, 111]},
    {"query": "Mô tả thuật toán Bellman-Ford tìm đường đi ngắn nhất trên đồ thị không có chu trình âm", "target_pages": [112, 113]},
    {"query": "Kiểm nghiệm thuật toán Bellman-Ford qua các vòng lặp K=1 và K=2", "target_pages": [113, 114, 115]},
    {"query": "Bảng 6.2 tổng hợp kết quả kiểm nghiệm các vòng lặp K theo thuật toán Bellman-Ford", "target_pages": [115, 116]},
    {"query": "Mô tả thuật toán Floy tìm đường đi ngắn nhất giữa tất cả các cặp đỉnh của đồ thị", "target_pages": [117, 118]},
    {"query": "Cài đặt thuật toán Floy trong ngôn ngữ C với mảng khoảng cách D và mảng vết S", "target_pages": [118, 119]},

    # 50 CÂU BỔ SUNG
    {"query": "Hình 1.1 mô hình mạng máy tính loại 1 biểu diễn bằng đơn đồ thị vô hướng", "target_pages": [7]},
    {"query": "Khái niệm cạnh khuyên và hình 1.3 mô tả giả đồ thị vô hướng", "target_pages": [8]},
    {"query": "Khái niệm đỉnh cô lập bậc 0 và đỉnh treo bậc 1 trên đồ thị vô hướng", "target_pages": [10]},
    {"query": "Định nghĩa đồ thị con liên thông và thành phần liên thông của đồ thị vô hướng", "target_pages": [11]},
    {"query": "Ví dụ 3 xác định các cạnh cầu và đỉnh trụ trên đồ thị vô hướng Hình 1.8", "target_pages": [12]},
    {"query": "Định lý 1 về tổng bán bậc vào bằng tổng bán bậc ra và bằng số cung của đồ thị có hướng", "target_pages": [13]},
    {"query": "Định lý 1 về điều kiện cần và đủ để đồ thị vô hướng định chiều được", "target_pages": [14]},
    {"query": "Hình 1.14 biểu diễn các đồ thị hai phía đầy đủ K2,3, K3,3 và K3,5", "target_pages": [15, 16]},
    {"query": "Hình 2.1 ví dụ ma trận kề biểu diễn đơn đồ thị vô hướng 6 đỉnh", "target_pages": [17]},
    {"query": "Ý nghĩa của phần tử a_ij^p trong lũy thừa ma trận kề A^p cho biết số đường đi qua p-1 đỉnh trung gian", "target_pages": [18]},
    {"query": "Hình 2.3 ma trận kề và ma trận trọng số của đồ thị có hướng 6 đỉnh", "target_pages": [19]},
    {"query": "Định nghĩa đồ thị thưa có số cạnh m <= 6n và lưu trữ bằng danh sách cạnh", "target_pages": [20]},
    {"query": "Hình 2.5 bảng danh sách cạnh đỉnh đầu đỉnh cuối của đồ thị có hướng", "target_pages": [21]},
    {"query": "Khai báo cấu trúc danh sách liên kết struct canh biểu diễn danh sách cạnh bằng con trỏ *Edge", "target_pages": [23, 24]},
    {"query": "Ví dụ mảng A[] 18 phần tử và mảng vị trí VT[6] lưu trữ danh sách kề", "target_pages": [25]},
    {"query": "Bài toán 6 viết chương trình lập ma trận kề cho bàn cờ vua 8x8 với quân mã", "target_pages": [28]},
    {"query": "Độ phức tạp tính toán O(n^2), O(n.m) và O(max(n,m)) của thuật toán DFS", "target_pages": [32]},
    {"query": "Bảng kiểm nghiệm từng bước duyệt DFS(1) và kết quả duyệt trên đồ thị 13 đỉnh", "target_pages": [33]},
    {"query": "Hàm Init() đọc ma trận kề từ file dothi.in và thiết lập mảng chuaxet trong DFS", "target_pages": [35]},
    {"query": "Mã nguồn C hàm DFS_Stack duyệt theo chiều sâu sử dụng mảng Stack", "target_pages": [35, 36]},
    {"query": "Hình 3.3 mô tả chi tiết các bước khởi tạo và vòng lặp thuật toán BFS(u)", "target_pages": [37]},
    {"query": "Kết quả thực hiện thuật toán BFS(3) trên ma trận kề 10 đỉnh trong file dothi.in", "target_pages": [39]},
    {"query": "Kiểm nghiệm thuật toán Duyet-TPLT tách đồ thị 13 đỉnh thành 2 thành phần liên thông", "target_pages": [42]},
    {"query": "Mã nguồn C hàm BFS và hàm main duyệt tất cả các thành phần liên thông", "target_pages": [43]},
    {"query": "Bảng 3.4 kiểm nghiệm thuật toán BFS(1) tìm đường đi ngắn nhất qua ít cạnh nhất", "target_pages": [46, 47]},
    {"query": "Hàm Duongdi() lần ngược mảng truoc[] để in đường đi từ đỉnh s đến đỉnh t", "target_pages": [47, 48]},
    {"query": "Mã C thủ tục Read_Data() và hàm BFS() kiểm tra tính liên thông mạnh", "target_pages": [51, 52]},
    {"query": "Mô tả thuật toán Duyet-Tru bằng cách gán chuaxet[u] = False và gọi DFS hoặc BFS", "target_pages": [53]},
    {"query": "Hình 3.10 mô tả thuật toán Duyet-Cau bằng cách loại bỏ tạm thời cạnh e", "target_pages": [56, 57]},
    {"query": "Vòng lặp hai phần tử for(u=1; u<n; u++) for(v=u+1; v<=n; v++) duyệt cạnh cầu trong C", "target_pages": [59, 60]},
    {"query": "Hình 3.12 ví dụ minh họa phép định chiều đồ thị vô hướng thành đồ thị có hướng liên thông mạnh", "target_pages": [61, 62]},
    {"query": "Bài tập 12 định nghĩa đỉnh thắt s của cặp đỉnh u và v trên đồ thị vô hướng", "target_pages": [67]},
    {"query": "Định lý 1 về điều kiện cần và đủ để đồ thị vô hướng và đồ thị có hướng là Euler", "target_pages": [68]},
    {"query": "Các bước chứng minh đồ thị vô hướng 13 đỉnh liên thông và mọi đỉnh đều có bậc chẵn", "target_pages": [69]},
    {"query": "Hình 4.3 mô tả thuật toán Euler-Cycle(u) sử dụng cấu trúc stack và mảng CE", "target_pages": [70]},
    {"query": "Mã C hàm Kiemtra() đếm số đỉnh bậc lẻ để xác định đồ thị có chu trình Euler", "target_pages": [72]},
    {"query": "Chứng minh đồ thị có hướng là nửa Euler với 2 đỉnh thỏa mãn deg+(1) - deg-(1) = 1", "target_pages": [74]},
    {"query": "Hàm Kiemtra() trong C kiểm tra điều kiện đồ thị có đúng 2 đỉnh bậc lẻ d == 2", "target_pages": [77]},
    {"query": "Định nghĩa đường đi Hamilton, chu trình Hamilton và đồ thị nửa Hamilton", "target_pages": [78, 79]},
    {"query": "Mã nguồn C hàm đệ quy quay lui Hamilton(int *B, int *C, int i)", "target_pages": [81, 82]},
    {"query": "Định lý về 6 khẳng định tương đương của một cây n đỉnh không có chu trình", "target_pages": [87, 88]},
    {"query": "Hình 5.2 mô tả thuật toán Tree-DFS(u) đệ quy kết nạp cạnh vào cây khung T", "target_pages": [88]},
    {"query": "Mã C hàm TREE_DFS() lưu các cạnh cây bao trùm vào mảng 2 chiều CBT[sc]", "target_pages": [90, 91]},
    {"query": "Mã C hàm TREE_BFS() sử dụng hàng đợi QUEUE tìm cây bao trùm của đồ thị", "target_pages": [93, 94]},
    {"query": "Danh sách các cạnh sắp xếp tăng dần theo trọng số để kiểm nghiệm thuật toán Kruskal", "target_pages": [97]},
    {"query": "Mã C thuật toán Kruskal với các hàm quản lý tập hợp Find(i) và Union(i, j)", "target_pages": [98, 99, 100]},
    {"query": "Mã nguồn C hàm PRIM() tìm đỉnh l có cạnh nhỏ nhất nối tập đỉnh s[top] với đỉnh chưa xét", "target_pages": [103, 104]},
    {"query": "Hình 6.1 mô tả thuật toán Dijkstra tìm đường đi ngắn nhất sử dụng nhãn tạm thời", "target_pages": [108]},
    {"query": "Mã C vòng lặp while(!final[t]) tìm đỉnh có nhãn tạm thời minp nhỏ nhất trong Dijkstra", "target_pages": [111, 112]},
    {"query": "Bài tập 2 bài toán tìm hành trình bay có khoảng cách bé nhất trong file hanhtrinh.in", "target_pages": [121, 122]}
]

In [7]:

def main():
    if not os.path.exists(PDF_PATH):
        raise FileNotFoundError(f"Không tìm thấy file: {PDF_PATH}")
    
    # 1. Trích xuất text và render ảnh
    print(f"[*] Đang trích xuất văn bản và render ảnh PDF (DPI={DPI})...")
    pdf = pdfium.PdfDocument(PDF_PATH)
    scale = DPI / 72.0
    doc_images = []
    doc_texts = []
    
    for page in pdf:
        doc_images.append(page.render(scale=scale).to_pil())
        text = page.get_textpage().get_text_range().strip()
        doc_texts.append(text if text else "Trang đồ thị hoặc sơ đồ toán học")
    total_pages = len(doc_images)
    print(f"[✓] Đã xử lý {total_pages} trang PDF.")
    
    # 2. Tạo embeddings thưa (BM25)
    print("[*] Đang khởi tạo mô hình BM25...")
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    bm25_doc_embeddings = list(bm25_model.embed(doc_texts))
    
    # 3. Cấu hình Qdrant Collection In-Memory
    print("[*] Đang cấu hình Qdrant In-Memory...")
    client = QdrantClient(":memory:")
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "colqwen": models.VectorParams(
                size=128,
                distance=models.Distance.DOT,
                multivector_config=models.MultiVectorConfig(
                    comparator=models.MultiVectorComparator.MAX_SIM
                )
            )
        },
        sparse_vectors_config={
            "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)
        }
    )
    
    # 4. Trích xuất ColQwen2.5 embeddings và ingest vào Qdrant
    print(f"[*] Nạp mô hình {MODEL_ID} lên {DEVICE}...")
    model = ColQwen2_5.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map=DEVICE
    ).eval()
    processor = ColQwen2Processor.from_pretrained(MODEL_ID)
    
    print(f"[*] Đang index {total_pages} trang vào Qdrant...")
    with torch.no_grad():
        for i in tqdm(range(total_pages), desc="Qdrant Ingestion", file=sys.stdout, ncols=80, leave=True):
            img = doc_images[i]
            batch_input = processor.process_images([img]).to(DEVICE)
            emb = model(**batch_input)[0].to("cpu").to(torch.float32).numpy()
    
            bm25_sparse = bm25_doc_embeddings[i]
            page_num = i + 1
    
            client.upsert(
                collection_name=COLLECTION_NAME,
                points=[
                    models.PointStruct(
                        id=page_num,
                        vector={
                            "colqwen": emb.tolist(),
                            "bm25": models.SparseVector(
                                indices=bm25_sparse.indices.tolist(),
                                values=bm25_sparse.values.tolist()
                            )
                        },
                        payload={"page": page_num}
                    )
                ]
            )
            del batch_input
            torch.cuda.empty_cache()
    
    # 5. Trích xuất Embeddings của câu truy vấn
    queries = [item["query"] for item in test_set]
    print(f"[*] Trích xuất vector cho {len(queries)} queries...")
    bm25_query_embeddings = list(bm25_model.embed(queries))
    
    query_colqwen_embeddings = []
    with torch.no_grad():
        for q in queries:
            q_input = processor.process_queries([q]).to(DEVICE)
            q_emb = model(**q_input)[0].to("cpu").to(torch.float32).numpy()
            query_colqwen_embeddings.append(q_emb)
            del q_input
            torch.cuda.empty_cache()
    
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    # 6. Bước 1: Cache kết quả thô Top 25 từ Qdrant vào RAM
    print("\n[*] Đang truy vấn Qdrant lưu cache Top 25 RRF vào RAM...")
    cached_retrievals = []
    for idx in tqdm(range(len(test_set)), desc="Caching Qdrant", file=sys.stdout, ncols=80, leave=True):
        q_col = query_colqwen_embeddings[idx]
        q_bm25 = bm25_query_embeddings[idx]

        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            prefetch=[
                models.Prefetch(query=q_col.tolist(), using="colqwen", limit=25),
                models.Prefetch(
                    query=models.SparseVector(
                        indices=q_bm25.indices.tolist(),
                        values=q_bm25.values.tolist()
                    ),
                    using="bm25",
                    limit=25
                )
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=25
        )

        cached_retrievals.append([
            (hit.payload["page"], 1.0 / (rank_idx + 1))
            for rank_idx, hit in enumerate(search_result.points)
        ])

    # 7. Bước 2 & 3: Quét tham số và đánh giá các chỉ số tại K=3
    param_grid = [(0.0, 0.0)] + list(itertools.product(forwards, backwards))
    total_queries = len(test_set)
    benchmark_records = []

    print(f"[*] Chạy Grid Search {len(param_grid)} cấu hình trên RAM (Tập trung K=3)...")
    for fwd, bwd in param_grid:
        h1, h3 = 0, 0
        rec_3_list = []
        rr_3_list = []

        for idx, item in enumerate(test_set):
            target_pages = set(item["target_pages"])
            raw_hits = cached_retrievals[idx]

            fused_scores = {}
            for p, base_score in raw_hits:
                fused_scores[p] = fused_scores.get(p, 0.0) + base_score
                if fwd > 0.0 and (p + 1 <= total_pages):
                    fused_scores[p + 1] = fused_scores.get(p + 1, 0.0) + (base_score * fwd)
                if bwd > 0.0 and (p - 1 >= 1):
                    fused_scores[p - 1] = fused_scores.get(p - 1, 0.0) + (base_score * bwd)

            ranked_pages = sorted(fused_scores.keys(), key=lambda x: fused_scores[x], reverse=True)

            # 1. Tính Hit Rank và MRR@3 (Cắt ngưỡng ở K=3)
            hit_rank = total_pages + 1
            for r_idx, p in enumerate(ranked_pages):
                if p in target_pages:
                    hit_rank = r_idx + 1
                    break

            if hit_rank == 1: h1 += 1
            if hit_rank <= 3: h3 += 1
            rr_3_list.append(1.0 / hit_rank if hit_rank <= 3 else 0.0)

            # 2. Tính  Recall@3 chuẩn Multi-Page
            top3_set = set(ranked_pages[:3])
            hits_in_top3 = len(top3_set.intersection(target_pages))
            
            rec_3_list.append(hits_in_top3 / len(target_pages))

        benchmark_records.append({
            "forward": fwd,
            "backward": bwd,
            "hit@1": (h1 / total_queries) * 100,
            "hit@3": (h3 / total_queries) * 100,
            "rec@3": (sum(rec_3_list) / total_queries) * 100,
            "mrr@3": sum(rr_3_list) / total_queries
        })

    baseline = benchmark_records[0]
    
    # Sắp xếp ưu tiên: MRR@3 -> Recall@3 -> Hit@3 -> Hit@1
    diffusion_records = sorted(
        benchmark_records[1:],
        key=lambda x: (x["mrr@3"], x["rec@3"], x["hit@3"], x["hit@1"]),
        reverse=True
    )

    # In bảng kết quả tại K=3
    print("\n" + "=" * 90)
    print(f"{'Forward':<8} | {'Backward':<8} | {'Hit@1 (%)':<10} | {'Hit@3 (%)':<10} | {'R@3 (%)':<9} | {'MRR@3':<6}")
    print("-" * 90)
    print(f"{baseline['forward']:<8.2f} | {baseline['backward']:<8.2f} | {baseline['hit@1']:<10.2f} | {baseline['hit@3']:<10.2f} | {baseline['rec@3']:<9.2f} | {baseline['mrr@3']:<6.4f} [BASELINE]")
    print("-" * 90)
    for res in diffusion_records[:10]:
        print(f"{res['forward']:<8.2f} | {res['backward']:<8.2f} | {res['hit@1']:<10.2f} | {res['hit@3']:<10.2f} | {res['rec@3']:<9.2f} | {res['mrr@3']:<6.4f}")
    print("=" * 90)

    # Lọc cấu hình tối ưu: Hit@1 không giảm quá 1% so với baseline, ưu tiên MRR@3 & Recall@3
    valid_configs = [r for r in diffusion_records if r["hit@1"] >= (baseline["hit@1"] - 1.0)]
    best = valid_configs[0] if valid_configs else diffusion_records[0]

    print(f"\n[★] Cấu hình tối ưu phục vụ cấp dữ liệu Top 3 cho VLM:")
    print(f"    - Forward / Backward : {best['forward']:.2f} / {best['backward']:.2f}")
    print(f"    - Hit@1              : {best['hit@1']:.2f}% (Baseline: {baseline['hit@1']:.2f}%)")
    print(f"    - Hit@3              : {best['hit@3']:.2f}% (Baseline: {baseline['hit@3']:.2f}%)")
    print(f"    - Recall@3           : {best['rec@3']:.2f}% (Baseline: {baseline['rec@3']:.2f}%)")
    print(f"    - MRR@3              : {best['mrr@3']:.4f} (Baseline: {baseline['mrr@3']:.4f})")

if __name__ == "__main__":
    main()

[*] Đang trích xuất văn bản và render ảnh PDF (DPI=150)...
[✓] Đã xử lý 125 trang PDF.
[*] Đang khởi tạo mô hình BM25...
[*] Đang cấu hình Qdrant In-Memory...
[*] Nạp mô hình vidore/colqwen2.5-v0.2 lên cuda...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/826 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/506 [00:00<?, ?it/s]

[*] Đang index 125 trang vào Qdrant...


Qdrant Ingestion:   0%|                                 | 0/125 [00:00<?, ?it/s]

[*] Trích xuất vector cho 116 queries...

[*] Đang truy vấn Qdrant lưu cache Top 25 RRF vào RAM...


Caching Qdrant:   0%|                                   | 0/116 [00:00<?, ?it/s]

[*] Chạy Grid Search 31 cấu hình trên RAM (Tập trung K=3)...

Forward  | Backward | Hit@1 (%)  | Hit@3 (%)  | R@3 (%)   | MRR@3 
------------------------------------------------------------------------------------------
0.00     | 0.00     | 77.59      | 90.52      | 75.72     | 0.8362 [BASELINE]
------------------------------------------------------------------------------------------
0.30     | 0.05     | 77.59      | 93.10      | 82.90     | 0.8434
0.35     | 0.05     | 77.59      | 92.24      | 82.90     | 0.8391
0.25     | 0.10     | 77.59      | 91.38      | 81.61     | 0.8391
0.30     | 0.10     | 77.59      | 91.38      | 82.47     | 0.8376
0.25     | 0.20     | 77.59      | 91.38      | 82.04     | 0.8376
0.25     | 0.05     | 77.59      | 91.38      | 81.18     | 0.8376
0.40     | 0.05     | 77.59      | 91.38      | 82.47     | 0.8362
0.48     | 0.05     | 77.59      | 91.38      | 82.47     | 0.8362
0.45     | 0.05     | 77.59      | 91.38      | 82.04     | 0.8362
0.48    